Este bloco cria os dados pra validação do recomendador, os dados foram gerados no mesmo formato que o utilizado pelo sistema original, estes livros não estão nas mesmas tabelas usadas pelo select do recomendador funcional.

In [2]:
# Este trecho cria uma tabela configurada exatamente como no select usado para o recomendador 

import sqlite3

#Conecta ao banco e carrega os dados da query pra variável ser usada depois
conn = sqlite3.connect("biblioteca.db")
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS basevalidacao (
    id INTEGER,
    title TEXT,
    authors TEXT,
    categoria TEXT,
    media_avaliacao REAL,
    total_downloads INTEGER
);

""")
conn.commit()

In [3]:
#Estre trecho cria os inserts para a tabela que foi criada

conn = sqlite3.connect("biblioteca.db")
cursor = conn.cursor()
cursor.execute("""
INSERT INTO basevalidacao (id, title, authors, categoria, media_avaliacao, total_downloads) VALUES
(1, 'O Diário de Anne Frank', 'Anne Frank', 'Biografia', 4.8, 1500),
(2, 'Na Natureza Selvagem', 'Jon Krakauer', 'Aventura', 4.5, 1200),
(3, 'O Tatuador de Auschwitz', 'Heather Morris', 'História', 4.7, 1300),
(4, 'Persépolis', 'Marjane Satrapi', 'Biografia', 4.6, 1100),
(5, 'A Menina Que Roubava Livros', 'Markus Zusak', 'Ficção Histórica', 4.9, 1600),
(6, 'O Pianista', 'Władysław Szpilman', 'Biografia', 4.8, 1400),
(7, 'A Sangue Frio', 'Truman Capote', 'Crime', 4.4, 1000),
(8, 'Wild Swans', 'Jung Chang', 'História', 4.6, 1150),
(9, 'Os Meninos da Rua Paulo', 'Ferenc Molnár', 'Ficção', 4.3, 900),
(10, 'Além do Bem e do Mal', 'Friedrich Nietzsche', 'Filosofia', 4.2, 850),
(11, 'O Diário de Myriam', 'Philippe Lobjois', 'Biografia', 4.5, 950),
(12, 'Um Caminho para a Liberdade', 'Jojo Moyes', 'Ficção Histórica', 4.7, 1250),
(13, 'A Menina da Montanha', 'Tara Westover', 'Biografia', 4.8, 1350),
(14, 'O Fundo é Apenas o Começo', 'Neal Shusterman', 'Ficção', 4.3, 800),
(15, 'O Último Trem para a Zona Verde', 'Paul Theroux', 'Viagem', 4.1, 750),
(16, 'O Homem que Amava os Cachorros', 'Leonardo Padura', 'Ficção Histórica', 4.6, 1100),
(17, 'O Sol é Para Todos', 'Harper Lee', 'Ficção', 4.9, 1700),
(18, '1984', 'George Orwell', 'Ficção Científica', 4.8, 1600),
(19, 'O Grande Gatsby', 'F. Scott Fitzgerald', 'Ficção', 4.4, 1050),
(20, 'Mataram a Cotovia', 'Harper Lee', 'Ficção', 4.9, 1650),
(21, 'O Apanhador no Campo de Centeio', 'J.D. Salinger', 'Ficção', 4.3, 950),
(22, 'A Revolução dos Bichos', 'George Orwell', 'Ficção', 4.7, 1300),
(23, 'O Velho e o Mar', 'Ernest Hemingway', 'Ficção', 4.5, 1200),
(24, 'Cem Anos de Solidão', 'Gabriel García Márquez', 'Ficção', 4.8, 1500),
(25, 'Dom Quixote', 'Miguel de Cervantes', 'Ficção', 4.6, 1400),
(26, 'Crime e Castigo', 'Fiódor Dostoiévski', 'Ficção', 4.7, 1350),
(27, 'Guerra e Paz', 'Liev Tolstói', 'Ficção', 4.8, 1450),
(28, 'O Conde de Monte Cristo', 'Alexandre Dumas', 'Ficção', 4.9, 1600),
(29, 'Orgulho e Preconceito', 'Jane Austen', 'Ficção', 4.8, 1550),
(30, 'Jane Eyre', 'Charlotte Brontë', 'Ficção', 4.7, 1400),
(31, 'O Morro dos Ventos Uivantes', 'Emily Brontë', 'Ficção', 4.6, 1300);
""")

conn.commit()
conn.close()

In [4]:
#df_validacao recebe a tabela criada para ser usada no recomendador

import sqlite3
import pandas as pd
conn = sqlite3.connect("biblioteca.db")

df_validacao = pd.read_sql_query("SELECT * FROM basevalidacao", conn)



In [5]:
import textwrap
from fuzzywuzzy import process

#Mostra as escolhas possíveis para o usuário
print("Escolha o critério de pesquisa:")
print("1 - Título")
print("2 - Autor")
print("3 - Categoria")

# Função que recomenda os livros
def recomendar_livros(entrada, criterio):
    
    entrada = str(entrada).strip().lower()
        # remove espaços e transforma a entrada em minúsculas
    if not entrada:
        return exibir_recomendacoes(
            "Nenhum termo foi inserido.\n\nAbaixo deixamos como recomendação nossos livros mais populares:\n",
            df_validacao.sort_values(by=["media_avaliacao", "total_downloads"], ascending=False).head(5)
        )
    if criterio == 1:  # Pesquisa por Título
    # Encontra a melhor correspondência para o título inserido 
        similaridade = process.extractOne(entrada, df_validacao['title'].str.lower(), score_cutoff=60)
        if similaridade:
                # Seleciona o livro que corresponde ao título encontrado
                livro = df_validacao[df_validacao['title'].str.lower() == similaridade[0]].iloc[0]
                # Pega a categoria do livro encontrado
                categoria = livro['categoria']
                # Busca recomendações de livros na mesma categoria
                recomendacoes = df_validacao[df_validacao['categoria'] == categoria].head(5)
        
                # Verifica se a similaridade é maior que 80%
                if similaridade[1] > 80:
                    return exibir_recomendacoes(
                    f"O livro '{similaridade[0].title()}' foi encontrado.\nConfira abaixo as recomendações de livros da mesma categoria:\n", 
                    recomendacoes
                    )
                else:
                    return exibir_recomendacoes(
                        f"Aqui estão os resultados da busca '{entrada}' por Título:\n", 
                    recomendacoes
                     )
    
    if criterio == 2:  #Pesquisa por Autor
        similaridade = process.extractOne(entrada, df_validacao['authors'].str.lower(), score_cutoff=60)
        if similaridade:
            if similaridade[1] >= 80:  #Formata a resposta diferente se a similaridade for maior ou igual a 80%
                return exibir_recomendacoes(f"Autor '{similaridade[0].title()}' encontrado.\nConfira abaixo as recomendações de obras desse autor com as melhores avaliações:\n", 
                                            df_validacao[df_validacao['authors'].str.lower().str.contains(similaridade[0], case=False, na=False)].head(5))
            else: #Responde de forma generalizada sobre o termo pesquisado
                return exibir_recomendacoes(f"Aqui estão os resultados da busca '{entrada}' por Autor:\n", 
                                            df_validacao[df_validacao['authors'].str.lower().str.contains(similaridade[0], case=False, na=False)].head(5))
    
    if criterio == 3:  #Pesquisa por Categoria
        similaridade = process.extractOne(entrada, df_validacao['categoria'].str.lower(), score_cutoff=60)
        if similaridade:
            recomendacoes = df_validacao[df_validacao['categoria'].str.lower() == similaridade[0]].head(5)
            return exibir_recomendacoes(f"Aqui estão os resultados da busca '{entrada}' por Categoria:\n", recomendacoes)
    
    recomendacoes = df_validacao.sort_values(by=["media_avaliacao", "total_downloads"], ascending=False).head(5)
    return exibir_recomendacoes(f"Nenhum resultado encontrado para '{entrada}'.\n\nAbaixo estão as recomendações dos livros mais populares em nossa biblioteca:\n", recomendacoes)

#Tratamentos das recomendações:
#Ordena os livros encontrados por ranking e remove os nomes duplicados (se houver)
df_validacao = df_validacao.sort_values(by=["media_avaliacao"], ascending=False).drop_duplicates(subset=["title"], keep="first")

#Remove livros com avaliação inferior a 3.7
df_validacao = df_validacao[df_validacao["media_avaliacao"] >= 3.70]

#Formata a resposta
def exibir_recomendacoes(mensagem, recomendacoes):
    recomendacoes["media_avaliacao"] = recomendacoes["media_avaliacao"].map(lambda x: round(x, 2) if pd.notna(x) else 0.0)
    relevancias = recomendacoes["media_avaliacao"].astype(float).tolist()
    
    return {
        "mensagem": mensagem,
        "recomendacoes": recomendacoes[['title', 'authors', 'categoria', 'media_avaliacao']].rename(
            columns={"title": "Título", "authors": "Autor", "categoria": "Categoria", "media_avaliacao": "Avaliação"}
        ).map(lambda x: x.title() if isinstance(x, str) else x)
    }

# Métricas de Avaliação

#Precision mede a precisão do teste

def precision(resultado, desejavel):
    # Verifica se há recomendações, se não retorna 0
    if resultado["recomendacoes"].empty:
        return 0 

    #trata os títulos recomendados e desejáveis para comparação (converte pra minúsculas e remove espaços) 
    recomendados = {titulo.lower().strip() for titulo in resultado["recomendacoes"]["Título"].dropna()} #dropna faz com que os valores nulos sejam removidos
    desejados = {titulo.lower().strip() for titulo in desejavel}

    # Verifica se há recomendações válidas, se não retorna 0
    if not recomendados: 
        return 0  

    #Calcula a precisão como a quantidade de itens desejáveis que foram recomendados
    return len(recomendados & desejados) / len(recomendados)

# Recall calcula quantos dos itens desejáveis foram recomendados
def recall(resultado, desejavel):
    # Verifica se há itens desejáveis, se não retorna 0
    if not desejavel:
        return 0 

    #Trata os títulos recomendados e desejáveis para comparação (converte para minúsculas e remove espaços)
    recomendados = {titulo.lower().strip() for titulo in resultado["recomendacoes"]["Título"].dropna()}
    desejados = {titulo.lower().strip() for titulo in desejavel}

    # Verifica se há recomendações, se não retorna 0
    if not recomendados:
        return 0

    #Calcula o Recall como a quantidade de itens desejáveis que foram recomendados 
    return len(recomendados & desejados) / len(desejados)

# F1-Score mede a precisão e a sensibilidade de um teste
def f1_score(resultado, desejavel):
    #Obtém a precisão
    p = precision(resultado, desejavel)
    #Obtém o Recall
    r = recall(resultado, desejavel)
    # Calcula o F1-Score como a média harmônica entre a precisão e o recall, evitando divisão por zero
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0


# Tabela verdade para testes
tabela_verdade = {
    1: {"entrada": "O Diário de Anne Frank", "desejavel": ["A Menina Da Montanha", "O Pianista", "O Diário de Anne Frank", "Persépolis", "O Diário De Myriam"]},
    2: {"entrada": "Harper Lee", "desejavel": ["O Sol é Para Todos", "Mataram a Cotovia"]},
    3: {"entrada": "Biografia", "desejavel": ["O Diário de Anne Frank", "O Pianista", "A Menina Da Montanha", "Persépolis", "O Diário De Myriam"]},
}

# Entrada do usuário
criterio_usuario = int(input("Digite o número correspondente: ").strip())
entrada_usuario = input("Digite o termo de pesquisa: ")
resultado = recomendar_livros(entrada_usuario, criterio_usuario)

desejavel = tabela_verdade.get(criterio_usuario, {}).get("desejavel", [])
precisao = precision(resultado, desejavel)
sensibilidade = recall(resultado, desejavel)
f1 = f1_score(resultado, desejavel)

# Mostra o retorno da busca e as recomendações
print("\n" + resultado["mensagem"])
if not resultado["recomendacoes"].empty:
    print(f"{'Título'.ljust(30)} {'Autor'.ljust(25)} {'Categoria'.ljust(20)} {'Avaliação'.ljust(30)}")
    for _, row in resultado["recomendacoes"].iterrows():
        titulo = textwrap.shorten(str(row["Título"]), width=30, placeholder="...").ljust(30)
        autor = textwrap.shorten(str(row["Autor"]), width=25, placeholder="...").ljust(25)
        categoria = textwrap.shorten(str(row["Categoria"]), width=20, placeholder="...").ljust(20)
        avaliacao = str(row["Avaliação"]).ljust(30)
        print(f"{titulo} {autor} {categoria} {avaliacao}")

print(f"\nPrecisão: {precisao:.2f}")
print(f"Recall: {sensibilidade:.2f}")
print(f"F1-Score: {f1:.2f}")

Escolha o critério de pesquisa:
1 - Título
2 - Autor
3 - Categoria

O livro 'O Diário De Anne Frank' foi encontrado.
Confira abaixo as recomendações de livros da mesma categoria:

Título                         Autor                     Categoria            Avaliação                     
O Diário De Anne Frank         Anne Frank                Biografia            4.8                           
A Menina Da Montanha           Tara Westover             Biografia            4.8                           
O Pianista                     Władysław Szpilman        Biografia            4.8                           
Persépolis                     Marjane Satrapi           Biografia            4.6                           
O Diário De Myriam             Philippe Lobjois          Biografia            4.5                           

Precisão: 1.00
Recall: 1.00
F1-Score: 1.00
